# PyTorch 可选扩展：从自动求导到模型训练

这本 Notebook 不属于原始深度学习笔记的主线。建议先完成：

1. `tryit.ipynb`
2. `numpy_studying.ipynb`
3. `math_foundations_for_ml.ipynb`
4. `LiMu_Deep_learning.ipynb`

本扩展把 Tensor 和自动求导连接到 `DataLoader`、训练循环、`nn.Module` 和多层感知机。


In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("PyTorch version:", torch.__version__)
print("计算设备:", device)


PyTorch version: 2.10.0
计算设备: mps


## 1. 非标量输出与上游梯度

`backward()` 本质上计算 vector-Jacobian product。输出是标量时，上游梯度默认为 1；输出是向量时，需要明确传入同形状向量，说明各输出如何组合。


In [2]:
x_vector = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y_vector = x_vector ** 2

upstream = torch.tensor([1.0, 0.5, 0.0])
y_vector.backward(gradient=upstream)

# d(x²)/dx = 2x，再逐元素乘 upstream。
print("VJP:", x_vector.grad)
print("理论结果:", 2 * x_vector.detach() * upstream)


VJP: tensor([2., 2., 0.])
理论结果: tensor([2., 2., 0.])


## 2. Dataset 与 DataLoader

`TensorDataset` 按第 0 维配对特征和标签；`DataLoader` 负责分批和打乱。


In [3]:
features = torch.linspace(-2, 2, 200).unsqueeze(1)
targets = 3 * features - 0.5 + 0.2 * torch.randn_like(features)

# TensorDataset 按第 0 维配对特征和标签；两者样本数必须相同。
dataset = TensorDataset(features, targets)
# DataLoader 每次给出 32 条样本；shuffle=True 每轮重新打乱顺序。
loader = DataLoader(dataset, batch_size=32, shuffle=True)

batch_X, batch_y = next(iter(loader))
print("一个 batch 的形状:", batch_X.shape, batch_y.shape)


一个 batch 的形状: torch.Size([32, 1]) torch.Size([32, 1])


## 3. 手写线性回归

参数更新放在 `torch.no_grad()` 中，因为更新动作本身不应该成为下一轮计算图的一部分。


In [4]:
w = torch.zeros((1, 1), requires_grad=True)
b = torch.zeros(1, requires_grad=True)
learning_rate = 0.05

for epoch in range(20):
    epoch_loss = 0.0

    for batch_X, batch_y in loader:
        predictions = batch_X @ w + b
        loss = ((predictions - batch_y) ** 2).mean()

        # 清除上一批梯度。
        if w.grad is not None:
            w.grad.zero_()
            b.grad.zero_()

        loss.backward()

        with torch.no_grad():
            w -= learning_rate * w.grad
            b -= learning_rate * b.grad

        epoch_loss += loss.item() * len(batch_X)

    if epoch % 5 == 0 or epoch == 19:
        print(f"epoch={epoch:02d}, loss={epoch_loss / len(dataset):.4f}")

print("学到的 w, b:", w.item(), b.item())


epoch=00, loss=6.6146
epoch=05, loss=0.0388
epoch=10, loss=0.0385
epoch=15, loss=0.0388
epoch=19, loss=0.0386
学到的 w, b: 2.9892306327819824 -0.4731961786746979


## 4. 使用 `nn.Module` 与优化器

- `nn.Module` 注册并管理参数；
- 损失函数定义优化目标；
- 优化器负责根据梯度更新参数。


In [5]:
# Linear(1,1) 表示每个样本 1 个输入特征、1 个输出；to 移动模型设备。
model = nn.Linear(in_features=1, out_features=1).to(device)
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.05)

for epoch in range(20):
    model.train()
    epoch_loss = 0.0

    for batch_X, batch_y in loader:
        batch_X = batch_X.to(device)
        batch_y = batch_y.to(device)

        predictions = model(batch_X)          # 1. 前向传播
        loss = loss_fn(predictions, batch_y)  # 2. 计算损失
        # zero_grad 清除上一批梯度；否则 PyTorch 默认继续累加。
        optimizer.zero_grad()                 # 3. 清空梯度
        loss.backward()                       # 4. 反向传播
        # step 根据当前参数的 .grad 和学习率执行一次更新。
        optimizer.step()                      # 5. 更新参数

        epoch_loss += loss.item() * len(batch_X)

    if epoch % 5 == 0 or epoch == 19:
        print(f"epoch={epoch:02d}, loss={epoch_loss / len(dataset):.4f}")


epoch=00, loss=6.3720
epoch=05, loss=0.0387
epoch=10, loss=0.0385


epoch=15, loss=0.0386
epoch=19, loss=0.0386


## 5. 训练模式与评估模式

`model.train()` 和 `model.eval()` 会影响 Dropout、BatchNorm 等层。评估时还要使用 `torch.no_grad()`；`eval()` 本身不会关闭梯度。


In [6]:
model.eval()
with torch.no_grad():
    sample = torch.tensor([[1.5]], device=device)
    prediction = model(sample)

print("x=1.5 的预测:", prediction.item())


x=1.5 的预测: 3.9878416061401367


## 6. 用多层感知机学习 XOR

多个线性层之间如果没有激活函数，整体仍等价于一次线性变换。XOR 不能被直线分开，因此需要 ReLU 等非线性激活。


In [7]:
xor_X = torch.randint(0, 2, (800, 2), dtype=torch.float32)
xor_y = (xor_X[:, 0].bool() ^ xor_X[:, 1].bool()).float().unsqueeze(1)
xor_X = xor_X + 0.08 * torch.randn_like(xor_X)

xor_loader = DataLoader(
    TensorDataset(xor_X, xor_y),
    batch_size=64,
    shuffle=True,
)

mlp = nn.Sequential(
    nn.Linear(2, 16),
    nn.ReLU(),
    nn.Linear(16, 1),  # 输出 logits，不在模型中手动加 sigmoid
).to(device)

# BCEWithLogitsLoss 合并 sigmoid 与二元交叉熵，数值更稳定。
xor_loss_fn = nn.BCEWithLogitsLoss()
xor_optimizer = torch.optim.Adam(mlp.parameters(), lr=0.02)

for epoch in range(60):
    mlp.train()
    for batch_X, batch_y in xor_loader:
        batch_X = batch_X.to(device)
        batch_y = batch_y.to(device)

        logits = mlp(batch_X)
        loss = xor_loss_fn(logits, batch_y)
        xor_optimizer.zero_grad()
        loss.backward()
        xor_optimizer.step()

mlp.eval()
with torch.no_grad():
    logits = mlp(xor_X.to(device))
    predicted = (torch.sigmoid(logits) >= 0.5).float().cpu()
    accuracy = (predicted == xor_y).float().mean()

print("XOR accuracy:", accuracy.item())


XOR accuracy: 1.0


## 7. 常见错误排查顺序

1. **shape**：输入、标签和输出的形状是否符合预期？
2. **device**：模型和数据是否在同一设备？
3. **dtype**：模型输入通常为 float，标签类型是否符合损失函数要求？
4. **loss**：输出是 logits 还是概率？是否重复使用 sigmoid？
5. **gradient**：是否忘记清零，或意外使用了 `detach()`？
6. **mode**：验证时是否使用 `eval()` 和 `no_grad()`？
7. **numerics**：学习率是否过大？是否出现 NaN 或 Inf？
8. **data**：特征和标签是否正确对齐？

## 练习

1. 给一个 `(2, 3, 4)` Tensor 分别沿三个 axis 求和，先写出结果 shape。
2. 删除 `keepdims=True`，观察按行归一化为什么失败。
3. 连续调用两次 `backward()`，验证梯度累加。
4. 删除 XOR 网络中的 ReLU，比较准确率。
5. 为线性回归增加验证集，并记录训练损失与验证损失。
